# 05 BERT Fine-tuning

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Understand how **BERT** (or a similar encoder) is **fine-tuned** for a downstream task (e.g. text classification)
- Use a **pre-trained encoder** + **classification head** and train on a small dataset (e.g. sentiment)
- See why we use BERT fine-tuning instead of training a classifier from scratch on text

---

## 🌍 Real life

**Where is this used?** BERT-style fine-tuning is used for **sentiment analysis**, **intent detection**, **named entity recognition**, and **question answering** in industry.

**In this notebook we use** a **pre-trained text encoder** (e.g. from Hugging Face or a small Keras NLP model) and add a **classification head** for sentiment. We use **BERT fine-tuning** (instead of training from scratch) **because** the encoder already learned language representations; we only train the head (or last layers) with **less data**.

**📌 Covers slide(s):** **13** — Multi-Head Attention, BERT. *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell. If `transformers` is not installed, we fall back to a simple LSTM-based classifier to show the same idea (encoder + head).

⏱ **Runtime:** This notebook may take 10–30 minutes on GPU (BERT/fine-tuning). Use a smaller subset of data or fewer epochs if needed (see unit README).

## Theory (short)

- **BERT:** Bidirectional Encoder Representations from Transformers. Pre-trained on large text; we **fine-tune** by adding a task-specific head (e.g. one Dense layer for classification) and training on our labels.
- **Fine-tuning:** Keep most of the pre-trained weights; train the new head and optionally the last few layers of the encoder with a small learning rate.
- **We use BERT (or a pre-trained encoder)** instead of training from scratch because it already captures syntax and semantics; we need less labeled data.
- **If Hugging Face is unavailable:** We use an LSTM encoder + Dense head to show the same pattern: encode text → classify.

## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow/Keras, NumPy. We use **IMDB sentiment** (or a small subset). Optional: Hugging Face `transformers` for a real BERT model.

**Dataset:** Real — IMDB (movie reviews, sentiment).

**Outputs:** Model summary, training loss/accuracy for 2 epochs, and test accuracy. One sentence: "We fine-tune an encoder + head instead of training from scratch."

## Step 1: Imports and load IMDB (we use IMDB for sentiment; in real life you'd use your own labels)

In [1]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

if HAS_TF:
    (x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=2000)
    maxlen = 80
    x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen, padding="post", truncating="post")
    x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen, padding="post", truncating="post")
    x_train, y_train = x_train[:2000], y_train[:2000]
    x_test, y_test = x_test[:500], y_test[:500]
    print("Train:", x_train.shape, "Test:", x_test.shape)
else:
    print("Install TensorFlow: pip install tensorflow")

Train: (2000, 80) Test: (500, 80)


## Step 2: Build encoder + classification head (we use this pattern instead of training from scratch; BERT would be the encoder)

In [2]:
if HAS_TF:
    model = keras.Sequential([
        keras.layers.Embedding(2000, 64, input_length=maxlen),
        keras.layers.LSTM(64, return_sequences=False),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.summary()
    print("\nSame idea as BERT fine-tuning: encoder (here LSTM) + classification head. With BERT, encoder is a transformer.")

/opt/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Same idea as BERT fine-tuning: encoder (here LSTM) + classification head. With BERT, encoder is a transformer.


## Step 3: Train (2 epochs)

In [3]:
if HAS_TF:
    history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=2, batch_size=64, verbose=1)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Test accuracy: %.4f" % acc)

Epoch 1/2


 1/32 ━━━━━━━━━━━━━━━━━━━━ 20s 664ms/step - accuracy: 0.6250 - loss: 0.6917

 4/32 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5840 - loss: 0.6924  

 7/32 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5625 - loss: 0.6929

10/32 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5547 - loss: 0.6929

...

25/32 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7099 - loss: 0.6784

28/32 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7078 - loss: 0.6768

31/32 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7060 - loss: 0.6751

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6900 - loss: 0.6557 - val_accuracy: 0.7020 - val_loss: 0.5744


Test accuracy: 0.7020


## 🌍 Real-World Worked Example — Sentiment Classification with Positional Encoding

**Industry context:**
- Twitter/X classifies 500M tweets/day using transformer-based sentiment models
- Amazon uses BERT-based models to classify product reviews automatically
- Customer service chatbots (Salesforce Einstein, Zendesk) use transformers for intent detection

We implement a **mini-Transformer encoder** for binary sentiment classification using toy data.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)
VOCAB_SIZE, SEQ_LEN, D_MODEL = 100, 10, 32

# ── Synthetic sentiment data ────────────────────────────────────────────────
# Positive: sequences with mostly high token IDs; Negative: low token IDs
def make_data(n=500):
    X, y = [], []
    for _ in range(n):
        label = np.random.randint(2)
        if label == 1:
            seq = np.random.randint(50, 100, SEQ_LEN)
        else:
            seq = np.random.randint(0, 50, SEQ_LEN)
        X.append(seq); y.append(label)
    return torch.tensor(np.array(X)), torch.tensor(y, dtype=torch.long)

X, y = make_data(800)
X_tr, y_tr = X[:640], y[:640]
X_te, y_te = X[640:], y[640:]

# ── Mini Transformer ────────────────────────────────────────────────────────
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
        # Learnable positional encoding
        self.pos_enc = nn.Embedding(SEQ_LEN, D_MODEL)
        encoder_layer = nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=4, dim_feedforward=64, batch_first=True)
        self.encoder  = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(D_MODEL, 2)
    def forward(self, x):
        pos = torch.arange(x.size(1)).unsqueeze(0)
        out = self.embed(x) + self.pos_enc(pos)
        out = self.encoder(out)
        return self.classifier(out.mean(1))  # pool over sequence

model   = MiniTransformer()
opt     = optim.Adam(model.parameters(), lr=5e-4)
loss_fn = nn.CrossEntropyLoss()
losses, accs = [], []

for epoch in range(80):
    model.train()
    loss = loss_fn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    acc = (model(X_te).argmax(1)==y_te).float().mean().item()
    accs.append(acc)
print(f"Sentiment classification accuracy: {acc*100:.1f}%")
print("Real-world: BERT achieves 94%+ on SST-2 sentiment benchmark.")

plt.plot(losses); plt.title("Transformer Training Loss — Sentiment Classifier")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.tight_layout(); plt.show()

## 🧩 Mini-exercise

**Try it:** Add a dropout layer (e.g. 0.3) between the encoder output and the Dense head, then retrain for 1 epoch. Does validation accuracy change?

---

## ✅ Summary

**What you did:** Built an encoder (LSTM) + classification head on IMDB sentiment; trained for 2 epochs. Same pattern as BERT fine-tuning: pre-trained encoder + task head.

**In real life you'd also:** Use Hugging Face `transformers` (e.g. `TFAutoModelForSequenceClassification`) for real BERT, tune learning rate, and use more data.

**The main idea:** BERT fine-tuning = use a pre-trained encoder + add a task head; we do it instead of training from scratch to use less labeled data.

**Next:** `04_transformer_attention` introduces attention; `06_gpt_text_generation` covers GPT-style generation.

## 📚 References & Further Reading

**Foundational Paper:**
- Vaswani et al. (2017) — [Attention Is All You Need](https://arxiv.org/abs/1706.03762) *(must-read)*

**Follow-up:**
- Devlin et al. (2018) — [BERT](https://arxiv.org/abs/1810.04805)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)
- Touvron et al. (2023) — [LLaMA](https://arxiv.org/abs/2302.13971)

**Interactive:** [The Illustrated Transformer by Jay Alammar](https://jalammar.github.io/illustrated-transformer/)

**State-of-the-Art:** GPT-4 uses transformer architecture with ~1 trillion parameters. LLaMA 3.1 (70B) runs on a single server.